# Test set and Train set extraction

The notebook takes care of extracting a test set from the cleaned combined dataframes.  

The test set contains around 12 job ads per month, with descriptions longer than 200 characters.

The test set ads are extracted from the train set to avoid data leakage and both new datasets are saved.

In [2]:
import pandas as pd
import json

with open("../data/clean_ads.json", encoding="utf-8") as f:
    clean_ads_df = pd.DataFrame(json.load(f))

print("Number of rows:", len(clean_ads_df))

Number of rows: 2729


In [3]:
# checking ads with short descriptions
shorty = clean_ads_df[clean_ads_df["description"].str.len() < 200]

print("Number of short descriptions:", len(shorty))
shorty.head()

Number of short descriptions: 3


,id,title,employer,municipality,region,occupation,published,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,"PLM Teamcenter Developer, Solution Architect. ..."
1621,30999468,Category Analyst,Cool Company Skandinavien AB,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2026-05-06T08:32:20,Must have +5 experience.
2546,31417403,Senior SAP Consultant,Acumant AB,Solna,Stockholms län,"Verksamhetskonsult, IT",2026-08-31T12:26:50,Experience: e.g. 7–10+ years of SAP experience...


In [4]:
# removing jobs with short desc
restricted_df = clean_ads_df[clean_ads_df["description"].str.len() >= 200]

print("Number of rows after removal of short ones", len(restricted_df))

Number of rows after removal of short ones 2726


In [5]:
# length buckets
restricted_df["len_bucket"] = pd.qcut(restricted_df["description"].str.len(), 3, labels=["short", "medium", "long"])

# extracting month
restricted_df["published"] = pd.to_datetime(restricted_df["published"])
restricted_df["month"] = restricted_df["published"].dt.month

# shuffling
restricted_df = restricted_df.sample(frac=1, random_state=42)

In [6]:
# help function to select samples in a balanced way by length for test set
def sample_balanced(d, n_per_bucket=4, seed=42):
    return d.groupby(["month", "len_bucket"], observed=True).sample(n=n_per_bucket, random_state=seed)

In [8]:
# creating test set
test_set = pd.concat([sample_balanced(restricted_df)])

print(f"Number of ads in the test set: {len(test_set)}\n")
print("Number of ads selected per month and length bucket:\n")
table = pd.crosstab(test_set["month"], test_set["len_bucket"], margins=True, margins_name="Total")
print(table)

Number of ads in the test set: 120

Number of ads selected per month and length bucket:

len_bucket  short  medium  long  Total
month                                 
1               4       4     4     12
2               4       4     4     12
3               4       4     4     12
4               4       4     4     12
5               4       4     4     12
6               4       4     4     12
7               4       4     4     12
8               4       4     4     12
9               4       4     4     12
10              4       4     4     12
Total          40      40    40    120
